In [ ]:
import mne
import numpy as np
from typing import Dict, List, Union, Optional

class EEGBandSeparator:
    """EEG数据分频处理接口，用于将脑电信号分解为不同频段"""
    
    def __init__(self, 
                 standard_bands: Optional[Dict[str, List[float]]] = None,
                 method: str = 'fir',
                 fir_design: str = 'firwin'):
        """
        初始化分频处理器
        
        参数:
            standard_bands: 标准频段定义，格式为{频段名称: [下限频率, 上限频率]}
                            若为None则使用默认频段
            method: 滤波方法，'fir'或'iir'
            fir_design: FIR滤波器设计方法，如'firwin'或'firwin2'
        """
        # 默认频段定义（单位：Hz）
        self.default_bands = {
            'delta': [0.5, 4],    #  delta波
            'theta': [4, 8],      #  theta波
            'alpha': [8, 13],     #  alpha波
            'lbeta': [13, 20],    #  lbeta波
            'hbeta': [20, 30],      #  theta波
            'gamma1': [30, 40],     #  alpha波
            'gamma2': [40, 49],
            'gamma3': [51, 60],      #  theta波
            'gamma4': [60, 70],     #  alpha波
            'gamma5': [70, 80],#  beta波
            'gamma6': [80, 90]     #  gamma波
        }
        
        # 使用用户定义的频段或默认频段
        self.bands = standard_bands if standard_bands is not None else self.default_bands
        
        # 滤波参数
        self.method = method
        self.fir_design = fir_design
        
        # 分频结果
        self.band_data: Dict[str, mne.io.Raw] = {}
        
    def get_available_bands(self) -> List[str]:
        """获取所有可用的频段名称"""
        return list(self.bands.keys())
    
    def add_custom_band(self, band_name: str, freq_range: List[float]) -> None:
        """
        添加自定义频段
        
        参数:
            band_name: 自定义频段名称
            freq_range: 频率范围，格式为[下限频率, 上限频率]
        """
        if len(freq_range) != 2 or freq_range[0] >= freq_range[1]:
            raise ValueError("频率范围必须是[下限, 上限]且下限小于上限")
        
        self.bands[band_name] = freq_range
        print(f"已添加自定义频段: {band_name} ({freq_range[0]}-{freq_range[1]}Hz)")
    
    def separate_bands(self, raw: mne.io.Raw, bands: Optional[List[str]] = None) -> Dict[str, mne.io.Raw]:
        """
        对EEG数据进行分频处理
        
        参数:
            raw: 预处理后的EEG数据（mne.io.Raw对象）
            bands: 需要提取的频段名称列表，若为None则提取所有频段
            
        返回:
            字典，键为频段名称，值为对应频段的EEG数据（mne.io.Raw对象）
        """
        # 确定需要处理的频段
        target_bands = bands if bands is not None else self.get_available_bands()
        
        # 检查目标频段是否有效
        for band in target_bands:
            if band not in self.bands:
                raise ValueError(f"未知频段: {band}，可用频段为: {self.get_available_bands()}")
        
        # 对每个频段进行带通滤波
        for band in target_bands:
            l_freq, h_freq = self.bands[band]
            print(f"正在提取 {band} 频段 ({l_freq}-{h_freq}Hz)...")
            
            # 复制原始数据并进行带通滤波
            band_raw = raw.copy().filter(
                l_freq=l_freq,
                h_freq=h_freq,
                method=self.method,
                fir_design=self.fir_design,
                skip_by_annotation='edge'
            )
            
            # 保存结果
            self.band_data[band] = band_raw
        
        print("分频处理完成")
        return self.band_data
    
    def plot_band_comparison(self, channel: str, tmin: float = 0, tmax: float = 10) -> None:
        """
        绘制指定通道在不同频段的信号对比图
        
        参数:
            channel: 要绘制的通道名称
            tmin: 起始时间（秒）
            tmax: 结束时间（秒）
        """
        if not self.band_data:
            raise RuntimeError("请先调用separate_bands()方法进行分频处理")
        
        if channel not in next(iter(self.band_data.values())).ch_names:
            raise ValueError(f"通道 {channel} 不存在于数据中")
        
        # 创建一个大的图形来展示所有频段
        import matplotlib.pyplot as plt
        n_bands = len(self.band_data)
        fig, axes = plt.subplots(n_bands, 1, figsize=(12, 2*n_bands), sharex=True)
        
        # 为每个频段绘制信号
        for i, (band, raw) in enumerate(self.band_data.items()):
            # 获取数据
            data, times = raw[channel, int(tmin*raw.info['sfreq']):int(tmax*raw.info['sfreq'])]
            data = data[0]  # 提取单通道数据
            
            # 绘制
            axes[i].plot(times, data)
            axes[i].set_title(f"{band} 频段 ({self.bands[band][0]}-{self.bands[band][1]}Hz)")
            axes[i].set_ylabel('Amplitude (V)')
            axes[i].grid(True, alpha=0.3)
        
        axes[-1].set_xlabel('Time (s)')
        plt.tight_layout()
        plt.show()    